# validate_bronze
**Purpose:** Verify all Sales tables in bronze ADLS are complete before running silver transformations.

Run this notebook after `pl-ingestion-sqlserver-to-bronze` and before `bronze_to_silver`.

Checks:
- ✅ All expected tables present
- ✅ Row counts (compare with SQL Server `sys.partitions`)
- ✅ Null counts per column
- ✅ Min / Max for numeric columns
- ✅ Summary report

In [ ]:
# Cell 1 — Widget + paths
dbutils.widgets.text("storage_account", "sadataeng260524dev")
STORAGE_ACCOUNT = dbutils.widgets.get("storage_account")
BRONZE = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"
SCHEMA = "Sales"
print(f"Storage account : {STORAGE_ACCOUNT}")
print(f"Bronze path     : {BRONZE}/{SCHEMA}/")

In [ ]:
%run ./autoload

In [ ]:
auth = bobydo.AdlsAuth(dbutils, spark)
auth.setup(STORAGE_ACCOUNT)

In [ ]:
# Cell 2 — Discover tables from bronze
files = dbutils.fs.ls(f"{BRONZE}/{SCHEMA}/")
tables = sorted([
    f.name.replace(".parquet", "").replace("/", "")
    for f in files
    if f.name.endswith(".parquet")
])
print(f"Found {len(tables)} tables in bronze/{SCHEMA}/")
for t in tables:
    print(f"  - {t}")

In [ ]:
# Cell 3 — Row count per table
from pyspark.sql import functions as F

results = []
print("=== ROW COUNTS ===")
for table in tables:
    df = spark.read.parquet(f"{BRONZE}/{SCHEMA}/{table}.parquet")
    row_count = df.count()
    col_count = len(df.columns)
    results.append({"table": table, "rows": row_count, "columns": col_count})
    print(f"  ✅ {table}: {row_count} rows, {col_count} cols")

In [ ]:
# Cell 4 — Null check per table
print("=== NULL COUNTS ===")
has_issues = False
for table in tables:
    df = spark.read.parquet(f"{BRONZE}/{SCHEMA}/{table}.parquet")
    null_counts = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()
    nulls = {k: v for k, v in null_counts.items() if v > 0}
    if nulls:
        print(f"  ⚠️  {table}: nulls found → {nulls}")
        has_issues = True
    else:
        print(f"  ✅ {table}: no nulls")
if not has_issues:
    print("\nAll tables: no nulls found ✅")

In [ ]:
# Cell 5 — Min / Max for numeric columns
from pyspark.sql.types import NumericType

print("=== MIN / MAX (numeric columns) ===")
for table in tables:
    df = spark.read.parquet(f"{BRONZE}/{SCHEMA}/{table}.parquet")
    num_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]
    if num_cols:
        aggs = []
        for c in num_cols:
            aggs += [F.min(c).alias(f"min_{c}"), F.max(c).alias(f"max_{c}")]
        result = df.select(aggs).collect()[0].asDict()
        print(f"  {table}: {result}")
    else:
        print(f"  {table}: no numeric columns")

In [ ]:
# Cell 6 — Summary report
import pandas as pd

summary_df = pd.DataFrame(results)
print("=== BRONZE VALIDATION SUMMARY ===")
print(summary_df.to_string(index=False))
print(f"\nTotal tables : {len(results)}")
print(f"Total rows   : {sum(r['rows'] for r in results):,}")
print()
print("Compare row counts with SSMS query:")
print("  SELECT s.name, t.name, p.rows")
print("  FROM sys.tables t")
print("  JOIN sys.schemas s ON t.schema_id = s.schema_id")
print("  JOIN sys.partitions p ON t.object_id = p.object_id")
print("  WHERE s.name = 'Sales' AND p.index_id IN (0,1)")
print()
print("✅ Proceed to bronze→silver only if all row counts match and nulls are acceptable.")